# 01 · Подготовка данных

**Пайплайн НИР — стадия 1 из 3.** Генерация обучающего датасета для edit-агента:

1. 1000 raw-графов (`dataset_gen`: 50 узлов, mixed-тип);
2. базовые LC-маршруты для каждого графа (3 комбо весов: demand / route / conn);
3. curriculum-аугментации по тирам (`copy_full → … → lc_clean`);
4. сохранение в папку датасета (`raw_graphs_1000.pkl`, `meta.csv`).

Дальше: **`02_agent_training.ipynb`** читает этот датасет.

> Вся логика — в `paper_experiments.training_lc`; ноутбук только вызывает и визуализирует.

In [ ]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## Параметры

Всё берётся из конфига `cfg/train/edit_scratch*.yaml` (единый источник — без
констант в ячейках). `PROFILE='smoke'` даёт крошечный датасет (10 графов) для
быстрой проверки пайплайна; `'full'` — полные 1000 графов статьи.

In [ ]:
from connectpt.routes_generator.paper_experiments.training_lc import (
    load_train_config, copytier_config)

PROFILE = 'smoke'   # 'smoke' (10 графов) | 'full' (1000 графов)

train_cfg = load_train_config('edit_scratch_smoke' if PROFILE == 'smoke' else 'edit_scratch')
ds = copytier_config(train_cfg)

print('профиль        :', PROFILE)
print('граф(ов)       :', ds.n_graphs, f'(по {ds.raw_n_nodes} узлов, тип {ds.raw_graph_type})')
print('тиры           :', list(ds.tiers))
print('маршруты       :', f'{ds.target_n_routes} шт, длина {ds.min_route_len}..{ds.max_route_len}')
print('папка датасета :', ds.new_dataset_dir)

## Генерация датасета

`build_copytier_dataset` детерминированно (сид из конфига) создаёт raw-графы,
строит LC-маршруты по каждому комбо весов и применяет curriculum-аугментации,
складывая всё в папку датасета. Повторный запуск с готовым датасетом — no-op
(если `dataset_gen.force_regen=false`).

In [ ]:
from connectpt.routes_generator.paper_experiments.training_lc import build_copytier_dataset

seconds = build_copytier_dataset(ds)
if seconds is not None:
    print(f'[timing] {seconds:.2f} с/граф  (всего {ds.n_graphs} графов)')
else:
    print('датасет уже готов — генерация пропущена')

## Загрузка и статистика по тирам

`TrainingDataModule` читает графы и seed-маршруты; `meta.csv` содержит тир и
метрики каждого augment-события. Ниже — усреднённые по тирам показатели
избыточности (`redun_*`) и загрузки перегонов до/после аугментации.

In [ ]:
from connectpt.routes_generator.training import TrainingDataModule

dm = TrainingDataModule(
    raw_graphs_path=ds.subset_pkl, lc_results_dir=ds.new_dataset_dir, device=device,
    min_route_len=ds.min_route_len, max_route_len=ds.max_route_len,
    target_n_routes=ds.target_n_routes).setup()
graphs, seed_routes = dm.graphs, dm.seed_routes
meta_df = pd.read_csv(ds.meta_csv)

print(f'графов={len(graphs)}  seed_routes={tuple(seed_routes.shape)}')
cols = ['applied_events', 'redun_before', 'redun_after', 'max_leg_use_after', 'd_un_after_pct']
present = [c for c in cols if c in meta_df.columns]
display(meta_df.groupby('tier')[present].mean().round(3).reindex(list(ds.tiers)))

## Визуализация: seed-маршруты по одному графу на тир

Для каждого тира берём первый граф и рисуем его сеть маршрутов (узлы —
географические позиции, рёбра — уличный граф, цветные пути — seed-маршруты).

In [ ]:
from connectpt.routes_generator.citygraph_dataset import STOP_KEY
from connectpt.routes_generator.reports.figures import plot_routes_grid

def graph_coords(g):
    """Позиции узлов + уличный граф одного графа — для рисовалки маршрутов."""
    return g[STOP_KEY].pos.detach().cpu(), g.street_adj.detach().cpu()

tier_of = dict(zip(meta_df['graph_index'], meta_df['tier']))
for tier in list(ds.tiers):
    idx = next((gi for gi, t in tier_of.items() if t == tier and gi < len(graphs)), None)
    if idx is None:
        continue
    coords, street_adj = graph_coords(graphs[idx])
    plot_routes_grid({f'{tier}  (граф #{idx})': seed_routes[idx]}, coords, street_adj)
    plt.show()

## Где лежит результат

Эти пути читает стадия 2 (`02_agent_training.ipynb`).

In [ ]:
print('raw-графы   :', ds.subset_pkl)
print('LC-маршруты :', ds.new_dataset_dir)
print('мета        :', ds.meta_csv)
print()
print('Далее: 02_agent_training.ipynb')